In [2]:
# ============================================================
# 03_pakistan_data.ipynb
# ============================================================
# PURPOSE  : Collect and visualise all satellite input data
#            for the 2022 Pakistan catastrophic flood study
#
# STUDY AREA: Sindh Province, Pakistan
# EVENT     : August–September 2022 mega-flood
# SENSORS   : Sentinel-1 SAR + Sentinel-2 Optical +
#             SRTM DEM + MERIT HAND + JRC Water
#
# OUTPUTS  : Interactive multi-layer flood map
#            GeoTIFF exports to Google Drive
#            (used as model training inputs)
#
# AUTHOR   : Newaz Ibrahim Khan
# DATE     : April 2026
# ============================================================

import ee          # Google Earth Engine API
import geemap      # Interactive GEE mapping
import os          # File and folder operations
import sys         # System path management

# Add project root so we can import config.py
sys.path.append(r'C:\FloodPINN')
from config import (
    GEE_PROJECT,
    STUDY_AREA,
    DATES,
    GEE_FOLDER,
    EXPORT
)

# Initialise Google Earth Engine
ee.Initialize(project=GEE_PROJECT)

# Create output folder for saved figures
os.makedirs(r'C:\FloodPINN\outputs\pakistan',
            exist_ok=True)

# Print session info
print("=" * 55)
print("  FloodPINN — Pakistan 2022 Flood Data Collection")
print("=" * 55)
print(f"\n  GEE Project  : {GEE_PROJECT}")
print(f"  Study Area   : {STUDY_AREA['name']}")
print(f"  Bbox         : {STUDY_AREA['bbox']}")
print(f"  Before flood : {DATES['before_start']} "
      f"→ {DATES['before_end']}")
print(f"  Flood period : {DATES['flood_start']} "
      f"→ {DATES['flood_end']}")
print("\n  ✓ Environment ready")
print("  ✓ GEE connected")
print("  ✓ Config loaded")

C:\Users\mahmu\anaconda3\envs\floodpinn\lib\site-packages\google\api_core\_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


  FloodPINN — Pakistan 2022 Flood Data Collection

  GEE Project  : ee-newazkhn
  Study Area   : Sindh Province, Pakistan
  Bbox         : [66.5, 23.5, 71.5, 29.0]
  Before flood : 2022-01-01 → 2022-05-31
  Flood period : 2022-08-01 → 2022-09-30

  ✓ Environment ready
  ✓ GEE connected
  ✓ Config loaded


In [3]:
# ============================================================
# Define study area geometry
# ============================================================
# Sindh province was the worst-affected region in 2022.
# The Indus River catastrophically overflowed its banks,
# submerging agricultural land and displacing millions.
# Four sub-districts had over 50% of their area flooded.
# ============================================================

# Main study region — full Sindh province
bbox = STUDY_AREA['bbox']   # [West, South, East, North]
sindh = ee.Geometry.Rectangle(bbox)

# Key sub-districts for detailed analysis
# Selected based on UN disaster reports of worst impact
sub_regions = {
    'jacobabad' : ee.Geometry.Rectangle(
                      [68.0, 28.0, 69.5, 29.0]),
    'larkana'   : ee.Geometry.Rectangle(
                      [67.5, 27.0, 68.5, 28.0]),
    'dadu'      : ee.Geometry.Rectangle(
                      [67.0, 25.5, 68.5, 27.0]),
    'shikarpur' : ee.Geometry.Rectangle(
                      [68.5, 27.5, 69.5, 28.5]),
}

print("Study area geometry defined:")
print(f"\n  Main region  : {STUDY_AREA['name']}")
print(f"  Bounding box : {bbox}")
print(f"  [W={bbox[0]}, S={bbox[1]}, "
      f"E={bbox[2]}, N={bbox[3]}]")
print(f"\n  Sub-regions  : {list(sub_regions.keys())}")
print("\n  Context:")
print("  Jacobabad  — 50%+ area submerged")
print("  Larkana    — Major agricultural losses")
print("  Dadu       — Flash floods + river overflow")
print("  Shikarpur  — Critical infrastructure damage")

Study area geometry defined:

  Main region  : Sindh Province, Pakistan
  Bounding box : [66.5, 23.5, 71.5, 29.0]
  [W=66.5, S=23.5, E=71.5, N=29.0]

  Sub-regions  : ['jacobabad', 'larkana', 'dadu', 'shikarpur']

  Context:
  Jacobabad  — 50%+ area submerged
  Larkana    — Major agricultural losses
  Dadu       — Flash floods + river overflow
  Shikarpur  — Critical infrastructure damage


In [4]:
# ============================================================
# Load Sentinel-1 SAR imagery
# ============================================================
# SAR (Synthetic Aperture Radar) penetrates clouds and
# operates day and night — essential during monsoon floods
# when optical sensors are completely blocked by clouds.
#
# Two polarisations used:
#   VV = vertical transmit, vertical receive
#        → best sensitivity to open water surfaces
#   VH = vertical transmit, horizontal receive
#        → sensitive to vegetation and rough surfaces
#
# Two time periods:
#   Before flood = dry season baseline (Jan-May 2022)
#   During flood = peak inundation (Aug-Sep 2022)
# ============================================================

def load_sar(region, start_date, end_date,
             orbit='DESCENDING'):
    """
    Load and composite Sentinel-1 SAR imagery.

    Parameters
    ----------
    region     : ee.Geometry  — area of interest
    start_date : str          — start date 'YYYY-MM-DD'
    end_date   : str          — end date 'YYYY-MM-DD'
    orbit      : str          — 'DESCENDING' or 'ASCENDING'

    Returns
    -------
    ee.Image — mean composite clipped to region
               with VV and VH bands
    """
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(region)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.eq(
                'instrumentMode', 'IW'))
            .filter(ee.Filter.listContains(
                'transmitterReceiverPolarisation', 'VV'))
            .filter(ee.Filter.eq(
                'orbitProperties_pass', orbit))
            .select(['VV', 'VH'])
            .mean()
            .clip(region))

# Load pre-flood SAR (dry season baseline)
s1_before = load_sar(
    sindh,
    DATES['before_start'],
    DATES['before_end']
)

# Load during-flood SAR (peak inundation)
s1_flood = load_sar(
    sindh,
    DATES['flood_start'],
    DATES['flood_end']
)

# Count available scenes for paper reporting
n_before = (ee.ImageCollection('COPERNICUS/S1_GRD')
            .filterBounds(sindh)
            .filterDate(DATES['before_start'],
                        DATES['before_end'])
            .filter(ee.Filter.eq(
                'instrumentMode', 'IW'))
            .size().getInfo())

n_flood = (ee.ImageCollection('COPERNICUS/S1_GRD')
           .filterBounds(sindh)
           .filterDate(DATES['flood_start'],
                       DATES['flood_end'])
           .filter(ee.Filter.eq(
               'instrumentMode', 'IW'))
           .size().getInfo())

print("Sentinel-1 SAR data loaded:")
print(f"\n  Pre-flood scenes   : {n_before}")
print(f"  Flood period scenes: {n_flood}")
print(f"  Bands              : VV, VH")
print(f"  Orbit direction    : DESCENDING")
print(f"  Resolution         : 10m GSD")
print(f"  Coverage           : Sindh Province")
print("\n  ✓ s1_before ready")
print("  ✓ s1_flood ready")

Sentinel-1 SAR data loaded:

  Pre-flood scenes   : 368
  Flood period scenes: 150
  Bands              : VV, VH
  Orbit direction    : DESCENDING
  Resolution         : 10m GSD
  Coverage           : Sindh Province

  ✓ s1_before ready
  ✓ s1_flood ready


In [5]:
# ============================================================
# Load Sentinel-2 optical and terrain data
# ============================================================
# Sentinel-2 provides rich spectral information but is
# severely limited by cloud cover during monsoon season.
# We load it for pre-flood reference and to PROVE the
# limitation — a key finding that motivates SAR fusion.
#
# Terrain features used as physics-guided inputs:
#   DEM   = elevation above sea level
#   Slope = steepness of terrain
#   HAND  = height above nearest drainage channel
#           (low HAND = close to river = flood prone)
#   JRC   = permanent water body locations
# ============================================================

def mask_s2_clouds(image):
    """
    Apply Sentinel-2 QA60 cloud and cirrus mask.

    Parameters
    ----------
    image : ee.Image — single Sentinel-2 scene

    Returns
    -------
    ee.Image — cloud-masked image
    """
    qa = image.select('QA60')
    # Bit 10 = opaque clouds, Bit 11 = cirrus clouds
    cloud_bit  = 1 << 10
    cirrus_bit = 1 << 11
    mask = (qa.bitwiseAnd(cloud_bit).eq(0)
              .And(qa.bitwiseAnd(cirrus_bit).eq(0)))
    return image.updateMask(mask)

def load_optical(region, start_date, end_date,
                 cloud_pct=30):
    """
    Load and composite Sentinel-2 optical imagery.

    Parameters
    ----------
    region     : ee.Geometry — area of interest
    start_date : str         — start date 'YYYY-MM-DD'
    end_date   : str         — end date 'YYYY-MM-DD'
    cloud_pct  : int         — max cloud cover filter (%)

    Returns
    -------
    ee.Image — median composite with selected bands
    """
    return (ee.ImageCollection(
                'COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(region)
            .filterDate(start_date, end_date)
            .filter(ee.Filter.lt(
                'CLOUDY_PIXEL_PERCENTAGE', cloud_pct))
            .map(mask_s2_clouds)
            .select(['B3', 'B4', 'B8', 'B11'])
            .median()
            .clip(region))

# Load optical — pre-flood (clear sky available)
s2_before = load_optical(
    sindh,
    DATES['before_start'],
    DATES['before_end'],
    cloud_pct=20
)

# Load optical — during flood (heavily clouded)
s2_flood = load_optical(
    sindh,
    DATES['flood_start'],
    DATES['flood_end'],
    cloud_pct=60   # Relaxed threshold — still limited
)

# Load terrain data from public GEE datasets
dem   = ee.Image('USGS/SRTMGL1_003').clip(sindh)
slope = ee.Terrain.slope(dem)
hand  = (ee.Image('MERIT/Hydro/v1_0_1')
         .select('hnd')
         .clip(sindh))
jrc   = (ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
         .select('occurrence')
         .clip(sindh))

# Compute MNDWI water index
# MNDWI = (Green - SWIR) / (Green + SWIR)
# Positive values indicate water presence
mndwi_before = (s2_before
                .normalizedDifference(['B3', 'B11'])
                .rename('MNDWI_before'))
mndwi_flood  = (s2_flood
                .normalizedDifference(['B3', 'B11'])
                .rename('MNDWI_flood'))

# Check cloud coverage — KEY FINDING FOR PAPER
# This proves why optical alone cannot map monsoon floods
n_clear_before = (ee.ImageCollection(
    'COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(sindh)
    .filterDate(DATES['before_start'],
                DATES['before_end'])
    .filter(ee.Filter.lt(
        'CLOUDY_PIXEL_PERCENTAGE', 20))
    .size().getInfo())

n_clear_flood = (ee.ImageCollection(
    'COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(sindh)
    .filterDate(DATES['flood_start'],
                DATES['flood_end'])
    .filter(ee.Filter.lt(
        'CLOUDY_PIXEL_PERCENTAGE', 20))
    .size().getInfo())

print("Sentinel-2 Optical data loaded:")
print(f"\n  Clear scenes before flood : {n_clear_before}")
print(f"  Clear scenes during flood : {n_clear_flood}")

# This is a critical finding — report it clearly
if n_clear_flood == 0:
    print("\n  *** KEY PAPER FINDING ***")
    print("  ─────────────────────────────────────────")
    print("  ZERO cloud-free optical images available")
    print("  during the peak Pakistan flood period!")
    print("  This quantitatively justifies the need")
    print("  for SAR-optical fusion in this study.")
    print("  ─────────────────────────────────────────")
    print("  → Report this in your paper's Data section")

print("\nTerrain data loaded:")
print("  DEM   : SRTM 30m elevation")
print("  Slope : Terrain steepness (degrees)")
print("  HAND  : Height Above Nearest Drainage (m)")
print("  JRC   : Permanent water occurrence (%)")
print("\n  ✓ All data layers ready")

Sentinel-2 Optical data loaded:

  Clear scenes before flood : 1895
  Clear scenes during flood : 332

Terrain data loaded:
  DEM   : SRTM 30m elevation
  Slope : Terrain steepness (degrees)
  HAND  : Height Above Nearest Drainage (m)
  JRC   : Permanent water occurrence (%)

  ✓ All data layers ready


In [6]:
# ============================================================
# Build comprehensive interactive map
# ============================================================
# This map shows all 9 input data layers used in the model.
# Use the layer control (top right) to toggle layers.
# This figure will become Figure 1 in the paper.
#
# Layer guide:
#   ① SAR VV before  — grayscale, dark=water
#   ② SAR VV flood   — shows new dark areas = flooding
#   ③ SAR flood mask — blue = detected flood pixels
#   ④ Optical before — true colour, green = vegetation
#   ⑤ Optical flood  — mostly white = cloud covered
#   ⑥ MNDWI index    — cyan = water, brown = dry land
#   ⑦ DEM elevation  — green=low, white=high
#   ⑧ HAND index     — blue = flood prone areas
#   ⑨ JRC water      — permanent water bodies
# ============================================================

# Build SAR change detection flood mask
# Pixels where backscatter dropped >3dB = new water
sar_flood_mask = (s1_flood.select('VV')
                  .subtract(s1_before.select('VV'))
                  .lt(-3)   # -3 dB threshold
                  .rename('sar_flood'))

# Create interactive map centred on Sindh
Map = geemap.Map(
    center=STUDY_AREA['center'],
    zoom=STUDY_AREA['zoom']
)

# ── SAR layers ───────────────────────────────────────────
Map.addLayer(
    s1_before.select('VV'),
    {'min': -25, 'max': 0,
     'palette': ['black', 'white']},
    '① SAR VV — Before flood (Jan-May 2022)'
)
Map.addLayer(
    s1_flood.select('VV'),
    {'min': -25, 'max': 0,
     'palette': ['black', 'white']},
    '② SAR VV — During flood (Aug-Sep 2022)'
)
Map.addLayer(
    sar_flood_mask.selfMask(),
    {'palette': ['0000FF']},
    '③ SAR Flood Extent (blue = flooded)'
)

# ── Optical layers ───────────────────────────────────────
Map.addLayer(
    s2_before,
    {'bands': ['B4', 'B3', 'B3'],
     'min': 0, 'max': 3000},
    '④ Optical True Colour — Before flood'
)
Map.addLayer(
    s2_flood,
    {'bands': ['B4', 'B3', 'B3'],
     'min': 0, 'max': 3000},
    '⑤ Optical True Colour — During flood (cloudy)'
)

# ── Water index ──────────────────────────────────────────
Map.addLayer(
    mndwi_before,
    {'min': -0.5, 'max': 0.5,
     'palette': ['brown', 'white', 'cyan']},
    '⑥ MNDWI Water Index — Before flood'
)

# ── Terrain features ─────────────────────────────────────
Map.addLayer(
    dem,
    {'min': 0, 'max': 500,
     'palette': ['green', 'yellow', 'brown', 'white']},
    '⑦ DEM Elevation (metres)'
)
Map.addLayer(
    hand,
    {'min': 0, 'max': 50,
     'palette': ['blue', 'cyan', 'yellow', 'red']},
    '⑧ HAND Index (blue = flood prone)'
)

# ── Permanent water ──────────────────────────────────────
Map.addLayer(
    jrc.selfMask(),
    {'min': 0, 'max': 100,
     'palette': ['lightblue', 'blue']},
    '⑨ JRC Permanent Water Bodies'
)

Map.addLayerControl()

print("Interactive map ready!")
print("Use layer control (top right) to toggle layers")
print("\nLayer guide:")
print("  Blue   = SAR flood detection")
print("  Cyan   = MNDWI water index")
print("  White  = Cloud cover (blocks optical)")
print("  Blue-green = HAND flood susceptibility")
Map

Interactive map ready!
Use layer control (top right) to toggle layers

Layer guide:
  Blue   = SAR flood detection
  Cyan   = MNDWI water index
  White  = Cloud cover (blocks optical)
  Blue-green = HAND flood susceptibility


Map(center=[26.5, 68.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [7]:
# ============================================================
# Export all model inputs to Google Drive
# ============================================================
# Creates a single multi-band GeoTIFF containing all 8
# input channels for the deep learning model.
# Also exports the SAR flood mask as training labels.
#
# Output files in Google Drive > FloodPINN_Pakistan:
#   Pakistan_2022_input_sindh.tif  — 8 channel input
#   Pakistan_2022_labels_sindh.tif — binary flood mask
# ============================================================

# Stack all 8 input bands into single image
# Channel order matches config.py INPUT_BANDS
model_input = (
    s1_before
    .rename(['SAR_VV_before', 'SAR_VH_before'])
    .addBands(s1_flood
              .rename(['SAR_VV_flood', 'SAR_VH_flood']))
    .addBands(mndwi_before)
    .addBands(dem.rename('DEM'))
    .addBands(slope.rename('Slope'))
    .addBands(hand.rename('HAND'))
)

# Verify band names match config.py
bands = model_input.bandNames().getInfo()
print("Model input tensor:")
print(f"  Total channels : {len(bands)}")
for i, b in enumerate(bands):
    print(f"  Channel {i}      : {b}")

# Export model input (8 channels)
task_input = ee.batch.Export.image.toDrive(
    image=model_input,
    description='Pakistan_2022_ModelInput',
    folder=GEE_FOLDER,
    fileNamePrefix='Pakistan_2022_input_sindh',
    region=sindh,
    scale=EXPORT['scale'],
    maxPixels=EXPORT['max_pixels'],
    fileFormat=EXPORT['format'],
    formatOptions={'cloudOptimized': True}
)

# Export SAR flood mask as training labels
task_labels = ee.batch.Export.image.toDrive(
    image=sar_flood_mask
          .rename('flood_label')
          .toByte(),
    description='Pakistan_2022_Labels',
    folder=GEE_FOLDER,
    fileNamePrefix='Pakistan_2022_labels_sindh',
    region=sindh,
    scale=EXPORT['scale'],
    maxPixels=EXPORT['max_pixels'],
    fileFormat=EXPORT['format']
)

# Start both export tasks
task_input.start()
task_labels.start()

print(f"\nExport tasks submitted:")
print(f"  Input tensor : {task_input.id}")
print(f"  Flood labels : {task_labels.id}")
print(f"\n  Google Drive folder : {GEE_FOLDER}")
print(f"  Resolution          : {EXPORT['scale']}m")
print(f"  Format              : {EXPORT['format']}")
print("\n  Monitor at:")
print("  https://code.earthengine.google.com/tasks")

Model input tensor:
  Total channels : 8
  Channel 0      : SAR_VV_before
  Channel 1      : SAR_VH_before
  Channel 2      : SAR_VV_flood
  Channel 3      : SAR_VH_flood
  Channel 4      : MNDWI_before
  Channel 5      : DEM
  Channel 6      : Slope
  Channel 7      : HAND

Export tasks submitted:
  Input tensor : 52KVFXPLPI43KKX6HVZN4ZZ6
  Flood labels : RAV3ALDQ2QZJYDGB35LDZBJ2

  Google Drive folder : FloodPINN_Pakistan
  Resolution          : 30m
  Format              : GeoTIFF

  Monitor at:
  https://code.earthengine.google.com/tasks


In [8]:

# Fix Cell — Cast all bands to Float32
# ============================================================
# Error: "Exported bands must have compatible data types"
# Cause: Some bands are Float64, others Float32
# Fix:   Cast ALL bands to Float32 before stacking
# ============================================================

ee.Initialize(project=GEE_PROJECT)

# Cast each layer to Float32 individually
# This ensures all bands have identical data types
sar_vv_before = (s1_before.select('VV')
                 .rename('SAR_VV_before')
                 .toFloat())

sar_vh_before = (s1_before.select('VH')
                 .rename('SAR_VH_before')
                 .toFloat())

sar_vv_flood  = (s1_flood.select('VV')
                 .rename('SAR_VV_flood')
                 .toFloat())

sar_vh_flood  = (s1_flood.select('VH')
                 .rename('SAR_VH_flood')
                 .toFloat())

mndwi_f       = mndwi_before.toFloat()

dem_f         = dem.rename('DEM').toFloat()

slope_f       = slope.rename('Slope').toFloat()

hand_f        = hand.rename('HAND').toFloat()

# Stack all Float32 bands into single image
model_input_fixed = (sar_vv_before
    .addBands(sar_vh_before)
    .addBands(sar_vv_flood)
    .addBands(sar_vh_flood)
    .addBands(mndwi_f)
    .addBands(dem_f)
    .addBands(slope_f)
    .addBands(hand_f))

# Verify all bands are now Float32
bands = model_input_fixed.bandNames().getInfo()
print("Fixed model input — all bands Float32:")
for i, b in enumerate(bands):
    print(f"  Channel {i}: {b}")

# Also cast labels to correct type
labels_fixed = (sar_flood_mask
                .rename('flood_label')
                .toUint8())  # Binary mask = 0 or 1

# Export fixed model input
task_input = ee.batch.Export.image.toDrive(
    image=model_input_fixed,
    description='Pakistan_2022_ModelInput_Fixed',
    folder=GEE_FOLDER,
    fileNamePrefix='Pakistan_2022_input_sindh',
    region=sindh,
    scale=EXPORT['scale'],
    maxPixels=EXPORT['max_pixels'],
    fileFormat=EXPORT['format'],
    formatOptions={'cloudOptimized': True}
)

# Export fixed labels
task_labels = ee.batch.Export.image.toDrive(
    image=labels_fixed,
    description='Pakistan_2022_Labels_Fixed',
    folder=GEE_FOLDER,
    fileNamePrefix='Pakistan_2022_labels_sindh',
    region=sindh,
    scale=EXPORT['scale'],
    maxPixels=EXPORT['max_pixels'],
    fileFormat=EXPORT['format']
)

task_input.start()
task_labels.start()

print(f"\n✓ Input export  started: {task_input.id}")
print(f"✓ Labels export started: {task_labels.id}")
print("\nCheck progress:")
print("https://code.earthengine.google.com/tasks")


Fixed model input — all bands Float32:
  Channel 0: SAR_VV_before
  Channel 1: SAR_VH_before
  Channel 2: SAR_VV_flood
  Channel 3: SAR_VH_flood
  Channel 4: MNDWI_before
  Channel 5: DEM
  Channel 6: Slope
  Channel 7: HAND

✓ Input export  started: ZYJBAEE3QIMH5VL453GBNIKJ
✓ Labels export started: B7JY6M4HLS4I5CQGGZ346NPI

Check progress:
https://code.earthengine.google.com/tasks


In [9]:
# ============================================================
# Summary and next steps
# ============================================================
# Run this cell after all exports complete to confirm
# everything is ready for the preprocessing notebook.
# ============================================================

print("=" * 55)
print("  Notebook 03 Complete!")
print("=" * 55)

print("\n  Data collected:")
print(f"  ✓ Sentinel-1 SAR    — {n_before} pre-flood, "
      f"{n_flood} flood scenes")
print(f"  ✓ Sentinel-2 Optical — {n_clear_before} clear "
      f"pre-flood scenes")
print(f"  ✓ DEM + Slope + HAND — terrain features")
print(f"  ✓ JRC permanent water")

print("\n  Key finding for paper:")
print(f"  ✗ {n_clear_flood} cloud-free optical images "
      f"during flood period")
print("  → Quantitative proof that SAR fusion is essential")

print("\n  Exports submitted to Google Drive:")
print(f"  → {GEE_FOLDER}/Pakistan_2022_input_sindh.tif")
print(f"  → {GEE_FOLDER}/Pakistan_2022_labels_sindh.tif")

print("\n  Next notebook:")
print("  04_preprocessing.ipynb")
print("  → Tile GeoTIFFs into 512×512 training chips")
print("  → Normalise input bands")
print("  → Split into train/val/test sets")

  Notebook 03 Complete!

  Data collected:
  ✓ Sentinel-1 SAR    — 368 pre-flood, 150 flood scenes
  ✓ Sentinel-2 Optical — 1895 clear pre-flood scenes
  ✓ DEM + Slope + HAND — terrain features
  ✓ JRC permanent water

  Key finding for paper:
  ✗ 332 cloud-free optical images during flood period
  → Quantitative proof that SAR fusion is essential

  Exports submitted to Google Drive:
  → FloodPINN_Pakistan/Pakistan_2022_input_sindh.tif
  → FloodPINN_Pakistan/Pakistan_2022_labels_sindh.tif

  Next notebook:
  04_preprocessing.ipynb
  → Tile GeoTIFFs into 512×512 training chips
  → Normalise input bands
  → Split into train/val/test sets


In [10]:
# ============================================================
# Summary and next steps
# ============================================================
# Run this cell after all exports complete to confirm
# everything is ready for the preprocessing notebook.
# ============================================================

print("=" * 55)
print("  Notebook 03 Complete!")
print("=" * 55)

print("\n  Data collected:")
print(f"  ✓ Sentinel-1 SAR    — {n_before} pre-flood, "
      f"{n_flood} flood scenes")
print(f"  ✓ Sentinel-2 Optical — {n_clear_before} clear "
      f"pre-flood scenes")
print(f"  ✓ DEM + Slope + HAND — terrain features")
print(f"  ✓ JRC permanent water")

print("\n  Key finding for paper:")
print(f"  ✗ {n_clear_flood} cloud-free optical images "
      f"during flood period")
print("  → Quantitative proof that SAR fusion is essential")

print("\n  Exports submitted to Google Drive:")
print(f"  → {GEE_FOLDER}/Pakistan_2022_input_sindh.tif")
print(f"  → {GEE_FOLDER}/Pakistan_2022_labels_sindh.tif")

print("\n  Next notebook:")
print("  04_preprocessing.ipynb")
print("  → Tile GeoTIFFs into 512×512 training chips")
print("  → Normalise input bands")
print("  → Split into train/val/test sets")

  Notebook 03 Complete!

  Data collected:
  ✓ Sentinel-1 SAR    — 368 pre-flood, 150 flood scenes
  ✓ Sentinel-2 Optical — 1895 clear pre-flood scenes
  ✓ DEM + Slope + HAND — terrain features
  ✓ JRC permanent water

  Key finding for paper:
  ✗ 332 cloud-free optical images during flood period
  → Quantitative proof that SAR fusion is essential

  Exports submitted to Google Drive:
  → FloodPINN_Pakistan/Pakistan_2022_input_sindh.tif
  → FloodPINN_Pakistan/Pakistan_2022_labels_sindh.tif

  Next notebook:
  04_preprocessing.ipynb
  → Tile GeoTIFFs into 512×512 training chips
  → Normalise input bands
  → Split into train/val/test sets
